# The Vibe Tax — McNemar's Test, Worked by Hand
### A reproducible walkthrough of the statistics behind the LiveCodeBench result

**What this notebook shows, end to end:**
1. Why the experiment needs a *paired* test (McNemar), not a two-proportion z-test.
2. McNemar's exact test **derived from first principles** — the coin-flip argument.
3. Our from-scratch formula **cross-checked against `scipy` and `statsmodels`** (proof the math is right).
4. The **real discordant counts** from our data plugged in.
5. The punchline: on the *same 219 paired problems*, changing **only the code extractor** turns a 
   "significant +8.7-point politeness tax" into a **null (p = 1.0)** — the effect was a scoring artifact.

Runs top-to-bottom on Google Colab with no setup (`scipy`/`statsmodels` are pre-installed).

## 0. Setup
Pure-standard-library math for our own implementation; `scipy` and `statsmodels` only to *check* it.

In [ ]:
from math import comb        # exact binomial coefficients, standard library
HAVE_SCIPY = HAVE_SM = False
try:
    import scipy; from scipy.stats import binomtest; HAVE_SCIPY = True
except ImportError: pass
try:
    import statsmodels; from statsmodels.stats.contingency_tables import mcnemar; HAVE_SM = True
except ImportError: pass
print('scipy:', scipy.__version__ if HAVE_SCIPY else 'not installed',
      '| statsmodels:', statsmodels.__version__ if HAVE_SM else 'not installed')
if not (HAVE_SCIPY and HAVE_SM):
    print('(on Google Colab both are pre-installed; the cross-check cell needs them)')

## 1. The design, and why the test must be *paired*

Every LiveCodeBench problem is sent to each model under **four framings** that wrap an *identical* 
problem statement — only the surrounding words differ:

| framing | wrapper |
|---|---|
| `agentic_terse` | *need `f(...)` for this: <problem>* |
| `agentic_casual` | *hey can you solve this? <problem>* |
| `webchat_detailed` | *Hi! Could you help me solve this problem? ... <problem> ... Thank you!* |
| `webchat_multilingual` | Chinese wrapper + <problem> |

For a given **(problem × model)** we get a pass/fail under `terse` and a pass/fail under `detailed`. 
Because these two share the *same problem and same model*, they are **matched pairs** — they are not 
independent samples. The right tool for paired binary outcomes is **McNemar's test**, which looks only 
at the pairs where the two conditions **disagree**.

## 2. McNemar from first principles

Lay every (problem × model) pair into a 2×2 table of *terse outcome* vs *detailed outcome*:

|                    | detailed **pass** | detailed **fail** |
|--------------------|:---:|:---:|
| **terse pass**     | a   | **b** |
| **terse fail**     | **c** | d   |

- **a** and **d** are *concordant* — both conditions agree (both pass, or both fail). They carry **no 
  information** about which framing is better, so McNemar ignores them.
- **b** = terse passed **but** detailed failed. **c** = detailed passed **but** terse failed. These 
  *discordant* pairs are the only evidence of a difference.

**The null hypothesis** \(H_0\): framing has no effect. Then for any pair that *disagrees*, it is equally 
likely to disagree in either direction — a fair coin. So under \(H_0\),

$$ b \sim \mathrm{Binomial}(n=b+c,\; p=\tfrac12). $$

The **exact two-sided p-value** is the probability of a split at least as lopsided as what we observed:

$$ p = 2\sum_{i=0}^{\min(b,c)} \binom{b+c}{i}\left(\tfrac12\right)^{b+c}, \quad\text{capped at } 1. $$

That is the entire test. The effect size we report is just the net discordance as a rate:

$$ \Delta = \frac{b-c}{n_\text{pairs}}\times 100\ \text{points.} $$

### 2a. Implement it ourselves (this is exactly `mcnemar_lcb.py :: exact_binom_two_sided`)

In [ ]:
def mcnemar_exact_p(b, c):
    """Two-sided exact McNemar p-value from the two discordant counts."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(comb(n, i) for i in range(k + 1)) / (2 ** n)   # P(X <= k)
    return min(1.0, 2 * tail)

def delta_points(b, c, n_pairs):
    return (b - c) / n_pairs * 100

# --- worked example, spelled out ---
b, c = 33, 14                      # (we'll justify these real numbers in section 3)
n = b + c
k = min(b, c)
print(f'discordant pairs: b={b} (terse>det), c={c} (det>terse), n=b+c={n}')
print(f'under H0, b ~ Binomial({n}, 0.5); observed k=min(b,c)={k}')
terms = [comb(n, i) for i in range(k + 1)]
print(f'sum of C({n},i) for i=0..{k}  =  {sum(terms)}   (out of 2^{n} = {2**n})')
print(f'one-tailed P(X<=k) = {sum(terms)/2**n:.6f}')
print(f'two-sided p        = {mcnemar_exact_p(b,c):.6f}')
print(f'effect size Delta  = {delta_points(b,c,219):+.1f} points (over 219 paired problems)')

### 2b. Proof the formula is correct — cross-check against two standard libraries

If our hand-rolled function agrees with `scipy.stats.binomtest` and `statsmodels`' `mcnemar` on a 
range of inputs, the math is right. (statsmodels with `exact=True` *is* the exact binomial McNemar.)

In [ ]:
assert HAVE_SCIPY and HAVE_SM, 'Run on Colab (or pip install scipy statsmodels) for the cross-check.'

def check(b, c):
    ours  = mcnemar_exact_p(b, c)
    scip  = binomtest(min(b, c), b + c, 0.5, alternative='two-sided').pvalue if (b+c) else 1.0
    table = [[10, b], [c, 10]]   # a,d are arbitrary — McNemar ignores them
    sm    = mcnemar(table, exact=True).pvalue
    return ours, scip, sm

print(f"{'b':>3} {'c':>3} | {'ours':>10} {'scipy':>10} {'statsmodels':>12}  match?")
all_ok = True
for b, c in [(33,14),(8,8),(46,24),(13,17),(7,7),(0,6),(45,22),(1,0),(100,60)]:
    o, s, m = check(b, c)
    ok = abs(o-s) < 1e-12 and abs(o-m) < 1e-12; all_ok &= ok
    print(f'{b:>3} {c:>3} | {o:>10.6f} {s:>10.6f} {m:>12.6f}  {"OK" if ok else "MISMATCH"}')
print('\nAll three implementations agree to machine precision:', all_ok)

They agree to machine precision on every input. Our `mcnemar_lcb.py` is computing the textbook exact 
McNemar test — no approximation, no chi-square correction (which matters because the discordant counts 
here are small).

## 3. Plug in the **real** data

These are the actual discordant counts from the experiment (3 models: ChatGPT + Claude + Codestral). 
The key comparison is `terse` vs `detailed`. The **medium** slice has the *same 219 paired problems* 
before and after we fixed the code extractor — so it is a clean apples-to-apples pair where **only the 
scorer changed.**

In [ ]:
# (b, c, n_pairs) for agentic_terse vs webchat_detailed, per slice.
# 'before' = naive code extractor; 'after' = robust extractor. Nothing else differs.
PRE = {  # naive extractor (124 problems; medium/hard/capable_medium unchanged vs post)
    'all':            (46, 24, 372),
    'medium':         (33, 14, 219),
    'hard':           (13, 10, 153),
    'capable_models': (45, 22, 248),
    'capable_medium': (32, 13, 146),
}
POST = {  # robust extractor (167 problems; easy added, so 'all'/'capable' n grow)
    'all':            (16, 21, 501),
    'medium':         ( 8,  8, 219),
    'hard':           ( 6, 10, 153),
    'capable_models': (13, 17, 334),
    'capable_medium': ( 7,  7, 146),
}

def stars(p): return '***' if p < .01 else ('*' if p < .05 else 'n.s.')
def show(title, D):
    print(title)
    print(f"  {'slice':16} {'b':>3} {'c':>3} {'n':>4} {'Delta':>7} {'p':>9}")
    for s,(b,c,n) in D.items():
        p = mcnemar_exact_p(b,c)
        print(f'  {s:16} {b:>3} {c:>3} {n:>4} {delta_points(b,c,n):>+6.1f} {p:>9.4f}  {stars(p)}')
    print()

show('BEFORE the extractor fix (naive) — terse vs detailed:', PRE)
show('AFTER the extractor fix (robust) — terse vs detailed:', POST)

### The headline, in one line

Look at the **medium** row — *the identical 219 paired problems* in both tables:

- **Before:** b=33, c=14  →  Δ = +8.7 points, **p ≈ 0.008** (significant: "polite framing costs correctness").
- **After:**  b=8,  c=8   →  Δ = **0.0 points, p = 1.000** (a perfect null).

We did not change the models, the prompts, or the problems — only how the model's code is extracted from 
its reply. The entire "politeness tax" lived in the scorer. Section 5 shows exactly why.

## 4. (Optional) Recompute from the full raw file

The cells above use the aggregate counts. To rebuild them from the 2,004 per-completion rows, load 
`lcb_scored.json` — upload it when prompted, or skip this cell if you just want the math.

In [ ]:
import json, io
scored = None
try:
    scored = json.load(open('lcb_scored.json'))
except FileNotFoundError:
    try:
        from google.colab import files
        up = files.upload()              # choose lcb_scored.json
        scored = json.load(io.BytesIO(next(iter(up.values()))))
    except Exception as e:
        print('No file loaded — skipping (the math above does not need it).', e)

if scored:
    def pass_map(cond, keep):
        return {(x['task_id'], x['model']): x['passed']
                for x in scored if x['level']==cond and keep(x)}
    def discordant(ca, cb, keep):
        a, b = pass_map(ca, keep), pass_map(cb, keep)
        ks = a.keys() & b.keys()
        bo = sum(a[k] and not b[k] for k in ks)
        co = sum(b[k] and not a[k] for k in ks)
        return bo, co, len(ks)
    med = lambda x: x.get('difficulty')=='medium'
    bo, co, n = discordant('agentic_terse','webchat_detailed', med)
    print(f'recomputed terse vs detailed, MEDIUM: b={bo}, c={co}, n={n}')
    print(f'  Delta={delta_points(bo,co,n):+.1f} pts, p={mcnemar_exact_p(bo,co):.4f}')
    print('  (matches the POST table above)')

## 5. Why the counts changed: the extraction artifact

The models were told to return **plain Python without markdown fences**. Polite/detailed prompts make 
the model *chatty* — it appends an explanation **after** the code. With no fences to mark where code 
ends, the original scorer handed the whole reply (code **+** English) to Python's parser, which throws a 
`SyntaxError` on the prose — so **correct code was scored as a failure.** Detailed prompts trigger more 
explanation, so they were failed more often. Here is a real completion from the run:

In [ ]:
completion = '''class Solution:
    def convertDateToBinary(self, date: str) -> str:
        year, month, day = date.split('-')
        return f"{bin(int(year))[2:]}-{bin(int(month))[2:]}-{bin(int(day))[2:]}"

This splits the date into year, month, and day, converts each part to an integer, then to binary.'''

import ast

def old_extract(text):          # naive: strip fence lines, keep everything else
    return '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```'))

def new_extract(text):          # robust: drop trailing lines until it parses
    lines = text.split('\n')
    while lines:
        src = '\n'.join(lines)
        try:
            ast.parse(src); return src
        except SyntaxError:
            lines.pop()
    return None

def compiles(src):
    try: ast.parse(src); return True
    except SyntaxError: return False

print('OLD extractor -> compiles?', compiles(old_extract(completion)))   # False: the prose breaks exec()
print('NEW extractor -> compiles?', compiles(new_extract(completion)))   # True: correct code recovered
print()
print('NEW extractor kept:')
print(new_extract(completion))

### The asymmetry that *was* the effect

Measured across all completions, the share whose extracted code even *compiled*:

| framing | naive extractor | robust extractor |
|---|---:|---:|
| terse | 88% | 100% |
| casual | 81% | 100% |
| multilingual | 77% | 100% |
| **detailed (polite)** | **73%** | 100% |

Detailed lost ~15 points *more than terse* to the parser — almost exactly the size of the phantom 
"+9-point politeness tax." Fix the extractor and the gap disappears, because the underlying pass rates 
were equal all along.

## 6. Takeaways for the walkthrough

1. **McNemar is just a coin-flip test on the pairs that disagree** — we derived it, implemented it in 
   five lines, and verified it against `scipy` and `statsmodels`.
2. **On the identical 219 paired problems, the result flips from p≈0.008 to p=1.000 when only the code 
   extractor changes.** The apparent "vibe tax" was a measurement artifact, not a property of the models.
3. **The honest finding is methodological:** how you *phrase* a complete coding request (terse, casual, 
   polite, or in another language) does not measurably change correctness on LiveCodeBench — and naive 
   code extraction can manufacture a convincing effect that isn't there.

*Reproducibility:* the test is `data/vibe_tax_lcb/mcnemar_lcb.py`; the fixed extractor is in 
`score_lcb.py :: _trim_to_compilable`; raw per-completion scores are in `lcb_scored.json`.